In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from datasets import load_dataset, DatasetDict

In [3]:
dataset = load_dataset(
    "Trendyol/Trendyol-Cybersecurity-Instruction-Tuning-Dataset",
    split="train"
)

dataset = dataset.select(range(8000))

README.md:   0%|          | 0.00/9.71k [00:00<?, ?B/s]

CyberSec-Dataset_escaped.jsonl: reconstructing file:   0%|          |  0.00B /  195MB            

CyberSec-Dataset_escaped.jsonl: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")

questions = dataset["user"]

# Filter out None values from the questions list
questions = [q for q in questions if q is not None]

embeddings = model.encode(
    questions,
    normalize_embeddings=True,
    show_progress_bar=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/250 [00:00<?, ?it/s]

In [11]:
def suggest_questions(current_question, top_k=5):

    query_embedding = model.encode(
        [current_question],
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        query_embedding,
        embeddings
    )[0]

    # Don't recommend the exact same question
    top_indices = np.argsort(scores)[::-1]

    suggestions = []

    for idx in top_indices:

        question = questions[idx]

        if question.lower().strip() == current_question.lower().strip():
            continue

        suggestions.append(question)

        if len(suggestions) == top_k:
            break

    return suggestions

In [12]:
suggest_questions(
    "What is phishing?"
)

['Discuss the importance of predictive phishing domain registration monitoring.',
 'Describe methods to identify adversary use of generative AI for phishing content creation.',
 'In what advanced evasions phishing uses LOLScripts, EDR monitors?',
 'Describe data‐science techniques to cluster phishing lure document metadata.',
 'How might stochastic model phishing success rates, mitigation optimizes?']

In [17]:
!ls

sample_data


In [19]:
import os
import numpy as np
import json

save_dir = "CyberShield_Question_Recommender"
os.makedirs(save_dir, exist_ok=True)

# Save embeddings
np.save(
    f"{save_dir}/question_embeddings.npy",
    embeddings
)

# Save questions
with open(
    f"{save_dir}/questions.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        questions,
        f,
        ensure_ascii=False,
        indent=2
    )

print("✅ Saved successfully")
print("Embeddings shape:", embeddings.shape)
print("Questions:", len(questions))

✅ Saved successfully
Embeddings shape: (7999, 384)
Questions: 7999


In [20]:
model.save(
    f"{save_dir}/sentence-transformer"
)

print("✅ Sentence Transformer saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Sentence Transformer saved


In [21]:
from sentence_transformers import SentenceTransformer
import numpy as np
import json

save_dir = "CyberShield_Question_Recommender"

# Load embedding model
model = SentenceTransformer(
    f"{save_dir}/sentence-transformer"
)

# Load embeddings
embeddings = np.load(
    f"{save_dir}/question_embeddings.npy"
)

# Load questions
with open(
    f"{save_dir}/questions.json",
    "r",
    encoding="utf-8"
) as f:
    questions = json.load(f)

print("✅ Model loaded")
print("Embeddings:", embeddings.shape)
print("Questions:", len(questions))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Model loaded
Embeddings: (7999, 384)
Questions: 7999


In [22]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def suggest_questions(current_question, top_k=5):

    query_embedding = model.encode(
        [current_question],
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        query_embedding,
        embeddings
    )[0]

    # Highest similarity first
    top_indices = np.argsort(scores)[::-1]

    suggestions = []

    for idx in top_indices:

        question = questions[idx]

        # Don't recommend identical question
        if question.lower().strip() == current_question.lower().strip():
            continue

        suggestions.append({
            "question": question,
            "score": float(scores[idx])
        })

        if len(suggestions) >= top_k:
            break

    return suggestions

In [23]:
results = suggest_questions(
    "What is phishing?",
    top_k=5
)

for i, result in enumerate(results, 1):
    print(
        f"{i}. {result['question']}"
        f"  | Similarity: {result['score']:.4f}"
    )

1. Discuss the importance of predictive phishing domain registration monitoring.  | Similarity: 0.6899
2. Describe methods to identify adversary use of generative AI for phishing content creation.  | Similarity: 0.6236
3. In what advanced evasions phishing uses LOLScripts, EDR monitors?  | Similarity: 0.6211
4. Describe data‐science techniques to cluster phishing lure document metadata.  | Similarity: 0.6140
5. How might stochastic model phishing success rates, mitigation optimizes?  | Similarity: 0.6123


In [24]:
import shutil

output_filename = "CyberShield_Question_Recommender.zip"
shutil.make_archive(
    base_name=output_filename.replace(".zip", ""),
    format='zip',
    root_dir='.',
    base_dir='CyberShield_Question_Recommender'
)

print(f"✅ Folder '{output_filename.replace(".zip", "")}' zipped successfully as '{output_filename}'")

✅ Folder 'CyberShield_Question_Recommender' zipped successfully as 'CyberShield_Question_Recommender.zip'
